# Causal Wizard &mdash; Results

Run **01-identification-and-estimation.ipynb** first to produce a `results.json`. This
notebook is pure presentation: every cell below reads from `results.json` (plus your
original config and data, for the plots that need the raw rows) and renders one section
of the results report &mdash; no further statistical computation happens here.

**Run this from inside the repo's `notebooks/` directory** so the local `causalwizard`
package is found automatically; in Colab, the next cell installs it from GitHub.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("causalwizard") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "causalwizard @ git+https://github.com/drawlinson/causal_wizard_app.git#subdirectory=notebooks"],
        check=True,
    )

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from causalwizard import config, identification, diagnostics, plotting, results_schema
from causalwizard.counterfactuals import SCENARIOS

## Inputs\n\nSame config/data files as notebook 1, plus the `results.json` it produced.

In [ ]:
config_path = "study-config.json"
data_path = "data.csv"
results_path = "results.json"

In [ ]:
cfg = config.load_config(config_path)
raw_df = pd.read_csv(data_path)
prepared = config.prepare_dataframe(cfg, raw_df)
train_df, test_df = config.train_test_split(prepared.df, cfg["question"]["splitTestPc"])

results = results_schema.load_results(results_path)
q = results["question"]
e = results["estimate"]
outcome_is_binary = q["outcomeType"] == "categorical"

## 1. Findings

In [ ]:
statement = diagnostics.findings_statement(
    q["treatment"], q["outcome"], q["treatmentIsContinuous"], q["outcomeType"], q["outcomeClass1Label"],
    q["targetUnits"], e["effect"],
)
print(f"Effect ({q['targetUnits'].upper()}): {e['effect']:.4g}")
print(statement)
print()
if e["accept"] is None:
    print("No validation tests were run for this estimator.")
elif e["accept"]:
    print(diagnostics.ACCEPT_TEXT)
else:
    print(diagnostics.REJECT_TEXT)

## 2. Outcomes plots

Outcomes by Sample Cohort always renders. Outcomes by Entity (scatter of observed vs.
predicted against treatment value) only applies to a numerical/continuous treatment
design. Outcomes over Time only applies when the study has a panel-data time variable.

In [ ]:
plotting.plot_outcomes_cohort(train_df, q["treatment"], q["outcome"], outcome_is_binary).show()

tp = e["trainPredictions"]
if tp is None:
    print("This estimator has no do-operator, so no predicted-outcome plots are available.")
else:
    if q["treatmentIsContinuous"]:
        plotting.plot_outcomes_entity(
            train_df, q["treatment"], q["outcome"], None,
            predicted=tp["predicted"], control_pred=tp["counterfactualControl"], treated_pred=tp["counterfactualTreated"],
            control_value=e["counterfactualControlValue"], treated_value=e["counterfactualTreatedValue"],
        ).show()
    if q["panelData"]:
        plotting.plot_outcomes_over_time(
            train_df, q["panelData"]["time"], q["panelData"]["entity"], q["outcome"], tp["predicted"]
        ).show()

## 3. Counterfactual outcomes table

Only available for regression models (CD+PO's own linear regression/GLM, or PD+FE) - the
only estimators with a do-operator. Note: for a categorical outcome, "Count"/"Sum"/"Mean"
are computed on the already-binary (0/1) encoded outcome, so "Sum" is the predicted count
of class-1 outcomes and "Mean" is the predicted class-1 probability - not a separate
per-category tally.

In [ ]:
cf = e["counterfactuals"]
if all(v is None for v in cf.values()):
    print("Not available for this estimator.")
else:
    rows = [
        {"Scenario": label, **(cf[key] if cf[key] else {"count": None, "sum": None, "mean": None})}
        for key, label in SCENARIOS
    ]
    display(pd.DataFrame(rows).set_index("Scenario"))

## 4. Refutation / validation, and held-out generalization

In [ ]:
print("Validation:", e["validation"])

gen = e["generalization"]
if gen is None:
    print("\nNo held-out generalization check available for this estimator.")
else:
    actual, predicted = np.array(gen["actual"]), np.array(gen["predicted"])
    if outcome_is_binary:
        cm = plotting.confusion_matrix(gen)
        print(f"\nAccuracy={cm['accuracy']:.3f}  Precision={cm['precision']:.3f}  "
              f"Recall={cm['recall']:.3f}  F1={cm['f1']:.3f}")
        print(cm)
    else:
        rmse = float(np.sqrt(np.mean((actual - predicted) ** 2)))
        mae = float(np.mean(np.abs(actual - predicted)))
        ss_res, ss_tot = float(np.sum((actual - predicted) ** 2)), float(np.sum((actual - actual.mean()) ** 2))
        r2 = 1 - ss_res / ss_tot if ss_tot else float("nan")
        print(f"\nR^2={r2:.3f}  RMSE={rmse:.4g}  MAE={mae:.4g}")
        plotting.plot_generalization_scatter(gen, treatment_col_is_binary=not q["treatmentIsContinuous"]).show()

## 5. Contingency table\n\nOnly meaningful for a grouped (Control/Treated) design - a continuous treatment has no such split.

In [ ]:
if q["treatmentIsContinuous"]:
    print("Not applicable - this study uses a continuous treatment, with no Control/Treated split.")
else:
    ct = results["sample"]["contingencyTable"]
    display(pd.DataFrame([ct["total"]], index=["All outcomes"]))
    if "by_outcome" in ct:
        rows = [{"Outcome": k, "Control": v["control"], "Treated": v["treated"]} for k, v in ct["by_outcome"].items()]
        display(pd.DataFrame(rows).set_index("Outcome"))

## 6. Positivity check\n\nOnly meaningful for propensity-based CD+PO estimators.

In [ ]:
pa = e["propensityAnalysis"]
if pa is None:
    print("Not a propensity-based estimator.")
else:
    plotting.plot_positivity(pa["distribution"]).show()

## 7. Covariate balance (love plot)

In [ ]:
if pa is None:
    print("Not a propensity-based estimator.")
else:
    plotting.plot_covariate_balance(pa["covariate_balance"]).show()

## 8. Assumptions

In [ ]:
print("General:")
for title, text in diagnostics.GENERAL_ASSUMPTIONS:
    print(f" - {title}: {text}")

print(f"\n{q['method'].upper()}-specific:")
method_assumptions = diagnostics.CDPO_ASSUMPTIONS if q["method"] == "cd+po" else diagnostics.PDFE_ASSUMPTIONS
for title, text in method_assumptions:
    print(f" - {title}: {text}")

if e["assumptions"]:
    print("\nSpecific to this estimand (from DoWhy):")
    for title, text in e["assumptions"].items():
        print(f" - {title}: {text}")

## 9. Causal diagram\n\nCD+PO studies only.

In [ ]:
if q["method"] != "cd+po":
    print("Causal diagram only applies to CD+PO studies.")
else:
    roles = {q["treatment"]: "treatment", q["outcome"]: "outcome"}
    for var in e["estimandVariables"]:
        roles[var] = e["estimandType"]
    plotting.plot_causal_diagram(cfg["graph"], roles)

## 10. Modelling statements

In [ ]:
for key, value in results["modellingStatements"].items():
    print(f"{key}: {value}")